In [ ]:
# tables for performance metrics

In [ ]:
import pickle
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score,  classification_report

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent.parent))
from config1 import CLASSIFICATION_DATA, get_predictions_classes, classes54


In [4]:
p_model_outputs = CLASSIFICATION_DATA / "4.MODEL_PREDICTIONS"



In [ ]:


name_mapping = {
    'Acacia': r'$\it{Acacia}$', 'Acer': r'$\it{Acer}$', 'Alnus': r'$\it{Alnus}$',
    'Amarantaceae': 'Amarantaceae', 'Artemisia': r'$\it{Artemisia}$', 'Betulaceae': r'$\it{Betulaceae}$',
    'Brassicaceae': 'Brassicaceae', 'Buxus': r'$\it{Buxus}$',
    'Carduus': r'$\it{Carduus}$',  'Caryophyllaceae': 'Caryophyllaceae',
    'Cichorioideae': 'Cichorioideae',
    'Cupressaceae': r'$\it{Cupressaceae}$', 'Cyperaceae': 'Cyperaceae',
    'Echium': r'$\it{Echium}$', 'Ericaceae': 'Ericaceae', 'Fagus': r'$\it{Fagus}$',
    'Fraxinus': r'$\it{Fraxinus}$', 'FraxinusExcelsior': r'$\it{Fraxinus~excelsior}$',     'FraxinusOrnus': r'$\it{Fraxinus~ornus}$',
    'Gallium': r'$\it{Galium}$',     'Helianthemum': r'$\it{Helianthemum}$', 'Hypericum': r'$\it{Hypericum}$', 'IlexAquifolium': r'$\it{Ilex~aquifolium}$',
    'IndetBlurry': 'Indeterminate (blurry)', 'IndetCovered': 'Indeterminate (covered)',
    'Juglans': r'$\it{Juglans}$', 'Lamiaceae': 'Lamiaceae','Liliaceae': 'Liliaceae',  'Lycopodium': r'$\it{Lycopodium}$',
    'Moraceae': 'Moraceae', 'Morphotype1' : 'Morphotype 1', 'Morphotype2' : "Morphotype 2", 
    'Myrtaceae': 'Myrtaceae', 'NonPollen': 'Non-pollen', 'Olea': r'$\it{Olea}$',
    'Other': 'Other', 'Phillyrea': r'$\it{Phillyrea}$',
    'Pinaceae': 'Pinaceae', 'Pistacia': r'$\it{Pistacia}$', 'Plantago': r'$\it{Plantago}$',
    'Platanus': r'$\it{Platanus}$', 'Poaceae': 'Poaceae', 'PopulusSp': r'$\it{Populus}$',
    'QuercusDeciduous': r'$\it{Quercus~deciduous}$', 'QuercusIlex': r'$\it{Quercus~ilex}$',
    'Ranunculaceae': 'Ranunculaceae', 'Rhamnus': r'$\it{Rhamnus}$', 'Rosaceae': 'Rosaceae',
    'Rumex': r'$\it{Rumex}$', 'Salix': r'$\it{Salix}$', 'Sanguisorba': r'$\it{Sanguisorba}$',
    'Tilia': r'$\it{Tilia}$', 'Ulmus': r'$\it{Ulmus}$', 'Urtica': r'$\it{Urtica}$',
    "ViburnumSambucusTp" : r'$\it{Viburnum / Sambucus}$',
    'VitisF': r'$\it{Vitis}$ fertile', 'VitisS': r'$\it{Vitis}$ sterile', 'XanthiumAmbrosia':  r'$\it{Xanthium / Ambrosia}$',
}




# Performances and accuracy - all classes

In [6]:
def get_acc(true, pred, cval_id) :

    acc= accuracy_score(true, pred)
    report_dict = classification_report(true, pred,  zero_division=0, output_dict=True)
    df_classif = pd.DataFrame(report_dict).transpose()
    df_classif["support"] = [int(el) for el in df_classif["support"]]
    df_classif["cval_id"]=cval_id
    df_classif = df_classif.reset_index().rename(columns={'index': 'classe'})
    df_classif["classe"] = [el.split("_")[0] for el in df_classif["classe"]]
    df_overall_acc = df_classif[df_classif["classe"].isin(["accuracy", 'weighted avg', "macro avg"])]
    df_prclass = df_classif[~df_classif["classe"].isin(["accuracy", 'weighted avg', "macro avg"])]
    df_prclass["classe2"] = [name_mapping[el] for el in df_prclass["classe"]]
    return acc, df_overall_acc, df_prclass






In [ ]:
# df ENV , using only environmental images
envref="env"


all_overall_acc = []
all_prclass = []
check_env=True
for idx in [1,2,3,4,5,6,7,8,9,10]:
    cval_id = f'fold{idx}'
    true, pred, filename =  get_predictions_classes(cval_id, p_model_outputs, classes54, envref=envref)
    accuracy, df_overall_acc, df_prclass = get_acc(true, pred, cval_id)
    all_overall_acc.append(df_overall_acc)
    all_prclass.append(df_prclass)

df_overall_acc_long = pd.concat(all_overall_acc, ignore_index=True)
df_prclass_long = pd.concat(all_prclass, ignore_index=True)

n_folds = df_prclass_long['cval_id'].nunique()
agg_df_env = df_prclass_long.groupby('classe')[['precision', 'recall', 'f1-score', 'support']].agg(['mean', 'std'])

agg_df_env[('precision', 'se')] = agg_df_env[('precision', 'std')] / np.sqrt(n_folds)
agg_df_env[('recall', 'se')] = agg_df_env[('recall', 'std')] / np.sqrt(n_folds)
agg_df_env[('f1-score', 'se')] = agg_df_env[('f1-score', 'std')] / np.sqrt(n_folds)
agg_df_env[('support', 'se')] = agg_df_env[('support', 'std')] / np.sqrt(n_folds)

agg_df_env = agg_df_env.drop(columns=[('precision', 'std'), ('recall', 'std'), ('f1-score', 'std'), ('support', 'std')])

agg_df_env.columns = ['_'.join(map(str, col)).strip() for col in agg_df_env.columns.values]
agg_df_env=agg_df_env.reset_index()

agg_df_env.head()


,classe,precision_mean,recall_mean,f1-score_mean,support_mean,precision_se,recall_se,f1-score_se,support_se
0,Acacia,0.000000,0.000000,0.000000,1.0,NaN,NaN,NaN,NaN
1,Acer,0.889318,0.943056,0.913723,8.9,0.027117,0.019027,0.019715,0.100000
2,Alnus,0.955909,0.941818,0.946711,10.5,0.019306,0.021776,0.014965,0.166667
3,Amarantaceae,0.957778,0.951389,0.950289,8.1,0.022819,0.027252,0.015940,0.100000
4,Artemisia,0.910000,0.841667,0.847857,3.3,0.060461,0.072913,0.060102,0.152753


In [ ]:
# df REF, using both reference and environmental images
envref="ref"


all_overall_acc = []
all_prclass = []
check_env=True
for idx in [1,2,3,4,5,6,7,8,9,10]:
    cval_id = f'fold{idx}'
    true, pred, filename =  get_predictions_classes(cval_id, p_model_outputs, classes54, envref=envref)
    accuracy, df_overall_acc, df_prclass = get_acc(true, pred, cval_id)
    all_overall_acc.append(df_overall_acc)
    all_prclass.append(df_prclass)

df_overall_acc_long = pd.concat(all_overall_acc, ignore_index=True)
df_prclass_long = pd.concat(all_prclass, ignore_index=True)

n_folds = df_prclass_long['cval_id'].nunique()
agg_df_ref = df_prclass_long.groupby('classe')[['precision', 'recall', 'f1-score', 'support']].agg(['mean', 'std'])

agg_df_ref[('precision', 'se')] = agg_df_ref[('precision', 'std')] / np.sqrt(n_folds)
agg_df_ref[('recall', 'se')] = agg_df_ref[('recall', 'std')] / np.sqrt(n_folds)
agg_df_ref[('f1-score', 'se')] = agg_df_ref[('f1-score', 'std')] / np.sqrt(n_folds)
agg_df_ref[('support', 'se')] = agg_df_ref[('support', 'std')] / np.sqrt(n_folds)

agg_df_ref = agg_df_ref.drop(columns=[('precision', 'std'), ('recall', 'std'), ('f1-score', 'std'), ('support', 'std')])

agg_df_ref.columns = ['_'.join(map(str, col)).strip() for col in agg_df_ref.columns.values]
agg_df_ref=agg_df_ref.reset_index()

agg_df_ref.head()


,classe,precision_mean,recall_mean,f1-score_mean,support_mean,precision_se,recall_se,f1-score_se,support_se
0,Acacia,1.000000,0.983333,0.990909,5.0,0.000000,0.016667,0.009091,0.149071
1,Acer,0.982025,0.991409,0.986659,93.1,0.003844,0.002137,0.002406,0.179505
2,Alnus,0.983854,0.976623,0.980110,55.7,0.004113,0.005436,0.003295,0.152753
3,Amarantaceae,0.998012,0.998002,0.998000,200.0,0.001097,0.001105,0.000624,0.149071
4,Artemisia,0.996887,0.996050,0.996452,126.8,0.002075,0.001770,0.001369,0.249444


In [10]:
#get df1 with all values o precision recall f1-scre *100 and rounded to 0.1
# 2SE
agg_df_env["precision_mean"] = (agg_df_env["precision_mean"]*100).round(1)
agg_df_env["recall_mean"] = (agg_df_env["recall_mean"]*100).round(1)
agg_df_env["f1-score_mean"] = (agg_df_env["f1-score_mean"]*100).round(1)
agg_df_env["precision_se"] = (agg_df_env["precision_se"]*100*2).round(1)
agg_df_env["recall_se"] = (agg_df_env["recall_se"]*100*2).round(1)
agg_df_env["f1-score_se"] = (agg_df_env["f1-score_se"]*100*2).round(1)
agg_df_env.head()

agg_df_ref["precision_mean"] = (agg_df_ref["precision_mean"]*100).round(1)
agg_df_ref["recall_mean"] = (agg_df_ref["recall_mean"]*100).round(1)
agg_df_ref["f1-score_mean"] = (agg_df_ref["f1-score_mean"]*100).round(1)
agg_df_ref["precision_se"] = (agg_df_ref["precision_se"]*100*2).round(1)
agg_df_ref["recall_se"] = (agg_df_ref["recall_se"]*100*2).round(1)
agg_df_ref["f1-score_se"] = (agg_df_ref["f1-score_se"]*100*2).round(1)
agg_df_ref.head()

,classe,precision_mean,recall_mean,f1-score_mean,support_mean,precision_se,recall_se,f1-score_se,support_se
0,Acacia,100.0,98.3,99.1,5.0,0.0,3.3,1.8,0.149071
1,Acer,98.2,99.1,98.7,93.1,0.8,0.4,0.5,0.179505
2,Alnus,98.4,97.7,98.0,55.7,0.8,1.1,0.7,0.152753
3,Amarantaceae,99.8,99.8,99.8,200.0,0.2,0.2,0.1,0.149071
4,Artemisia,99.7,99.6,99.6,126.8,0.4,0.4,0.3,0.249444


In [11]:
# for all classes represented, sorted in alphabetcic (a->z), write a latex line with class name, precision+-SE of df1, recall+-SE of df1, f1-score+-SE of df1, support of df1, precision+-SE of df2, recall+-SE of df2, f1-score+-SE of df2, support of df2, with mean and SE (2SE) for each metric

latex_rows = []

print('Full dataset (ref and env) & environment only \\\\')
print(("Classe & Precision (mean ± 2SE) & Recall (mean ± 2SE) & F1-score (mean ± 2SE) & Support (mean ± 2SE) & Precision (mean ± 2SE) & Recall (mean ± 2SE) & F1-score (mean ± 2SE) & Support (mean ± 2SE) \\\\"))
for _, row1 in agg_df_ref.iterrows():
    class_name = row1["classe"]
    try:
        row2 = agg_df_env[agg_df_env["classe"] == class_name].iloc[0]  # Get the corresponding row in df2
        line = f"{class_name} & {row1['precision_mean']}+/-{row1['precision_se']} & {row1['recall_mean']}+/-{row1['recall_se']} & {row1['f1-score_mean']}+/-{row1['f1-score_se']} & {row1['support_mean']} & {row2['precision_mean']}+/-{row2['precision_se']} & {row2['recall_mean']}+/-{row2['recall_se']} & {row2['f1-score_mean']}+/-{row2['f1-score_se']} & {row2['support_mean']} \\\\"

    except IndexError:
        line = f"{class_name} & {row1['precision_mean']}+/-{row1['precision_se']} & {row1['recall_mean']}+/-{row1['recall_se']} & {row1['f1-score_mean']}+/-{row1['f1-score_se']} & {row1['support_mean']} & NA & NA & NA & NA \\\\"

    latex_rows.append(line)
latex_table = "\n".join(latex_rows)
print(latex_table)

Full dataset (ref and env) & environment only \\
Classe & Precision (mean ± 2SE) & Recall (mean ± 2SE) & F1-score (mean ± 2SE) & Support (mean ± 2SE) & Precision (mean ± 2SE) & Recall (mean ± 2SE) & F1-score (mean ± 2SE) & Support (mean ± 2SE) \\
Acacia & 100.0+/-0.0 & 98.3+/-3.3 & 99.1+/-1.8 & 5.0 & 0.0+/-nan & 0.0+/-nan & 0.0+/-nan & 1.0 \\
Acer & 98.2+/-0.8 & 99.1+/-0.4 & 98.7+/-0.5 & 93.1 & 88.9+/-5.4 & 94.3+/-3.8 & 91.4+/-3.9 & 8.9 \\
Alnus & 98.4+/-0.8 & 97.7+/-1.1 & 98.0+/-0.7 & 55.7 & 95.6+/-3.9 & 94.2+/-4.4 & 94.7+/-3.0 & 10.5 \\
Amarantaceae & 99.8+/-0.2 & 99.8+/-0.2 & 99.8+/-0.1 & 200.0 & 95.8+/-4.6 & 95.1+/-5.5 & 95.0+/-3.2 & 8.1 \\
Artemisia & 99.7+/-0.4 & 99.6+/-0.4 & 99.6+/-0.3 & 126.8 & 91.0+/-12.1 & 84.2+/-14.6 & 84.8+/-12.0 & 3.3 \\
Betulaceae & 99.2+/-0.3 & 99.1+/-0.4 & 99.1+/-0.2 & 200.0 & 95.4+/-1.8 & 93.8+/-4.0 & 94.5+/-2.2 & 19.6 \\
Brassicaceae & 99.1+/-0.5 & 99.5+/-0.4 & 99.3+/-0.4 & 124.6 & 95.2+/-2.9 & 96.7+/-2.9 & 95.9+/-2.5 & 18.1 \\
Buxus & 98.6+/-0.6 & 99